# Adapt the Precision Miner

The built-in miner in notebook 05 finds data sources. Here we define a different extraction task—funding sources—without writing a new workflow class. A miner returns a variable-length list of **items to verify**, whereas a classifier returns labels that can be compared across papers.


In [ ]:
import sys
from pathlib import Path

if (Path.cwd() / "notebooks" / "epilens_nb.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "notebooks"))
import epilens_nb as nb


## Specify the question once

A `TaskSpec` gives the miner fixed retrieval questions. The workflow supplies the extraction prompt and JSON schema, then applies the same specification to each paper.


In [ ]:
from epilens.workflows.registry import TaskSpec, build_miner_config_from_spec

funding_spec = TaskSpec.model_validate({
    "key": "find_funding_sources",
    "kind": "miner",
    "label": "Funding sources",
    "description": "Grants and organisations that funded the study.",
    "top_k": 2,
    "retrieval_templates": [
        "Who funded this study, and which grants are named?",
        "Which organisations supported the work?",
    ],
})
funding_config = build_miner_config_from_spec(funding_spec)
print(funding_config.retrieval_templates)


## Run one illustrative case

`StaticRetriever` supplies one known excerpt, and `FixedJSONGenerator` supplies a scripted answer. This runs offline and shows the result shape; the fixed answer is **not** a model extraction.


In [ ]:
from epilens.schemas import PaperMetadata, SearchResult
from epilens.workflows import PrecisionMiner

excerpt = "This work was supported by the National Health Research Fund under grant NHRF-42."
chunk = SearchResult(
    id="funding-1", paper_id="paper_funding", section_title="Funding",
    section_type="Funding", text=excerpt, similarity_score=0.9,
)
miner = PrecisionMiner(
    retriever=nb.StaticRetriever([chunk]),
    generator=nb.FixedJSONGenerator({
        "description": "The paper names one funding source.",
        "items": [{
            "name": "National Health Research Fund",
            "url": None,
            "explanation": "The funding statement names this organisation and grant NHRF-42.",
            "raw_text": excerpt,
        }],
    }),
    config=funding_config,
)
result = miner.run_detailed(
    "paper_funding",
    metadata=PaperMetadata(title="Example funding statement", abstract="A short example paper."),
)
for item in result.result.items:
    print(f"{item.name}: {item.explanation}")
print("Source excerpt:", result.relevant_chunks[0].text)


## What to inspect

Compare each item and its `raw_text` with `result.relevant_chunks` before using it. The schema checks the fields, not whether the extraction is true or complete. With a real generator, `run_detailed` also retains the prompt, raw response, and any generator provenance. The same spec can be saved as JSON and passed to `epilens precision-miner --task-file`.
